# 03 — Paired Wilcoxon signed-rank tests

Runs two-sided paired Wilcoxon tests. Very small p-values should be reported as `p < .001`, not `p = .000`.

In [ ]:
from pathlib import Path
import pandas as pd
from scipy.stats import wilcoxon

DATA = Path('data/dar_naturalized_data_70.csv')
OUT = Path('outputs'); OUT.mkdir(exist_ok=True)
measures = {
    'Prompting density': ('Pre_Density', 'Post_Density'),
    'Mean prompt length': ('Pre_Length', 'Post_Length'),
    'AI-suggestion rejection rate': ('Pre_Rejection', 'Post_Rejection'),
    'Stance marking': ('Pre_Stance', 'Post_Stance'),
    'Lexical agency': ('Pre_Lexical', 'Post_Lexical'),
    'Voice consistency': ('Pre_Consistency', 'Post_Consistency'),
    'Critical dialogue': ('Pre_Critical', 'Post_Critical'),
}
df = pd.read_csv(DATA)
rows = []
for name, (pre, post) in measures.items():
    pair = df[[pre, post]].dropna()
    delta = pair[post] - pair[pre]
    test = wilcoxon(pair[post], pair[pre], alternative='two-sided', method='approx')
    rows.append({'Measure': name, 'N_pairs': len(pair), 'W': float(test.statistic),
                 'Z_abs_approx': abs(float(test.zstatistic)), 'p_value': float(test.pvalue),
                 'Increased': int((delta > 0).sum()), 'Decreased': int((delta < 0).sum()),
                 'Ties': int((delta == 0).sum())})
result = pd.DataFrame(rows)
result.to_csv(OUT / 'wilcoxon_tests.csv', index=False)
display(result)
print('Report p-values below .001 as p < .001, not p = .000.')